# Spill detection — YOLOv8 training template

This is a credential-free starter template based on the supplied notebook's YOLOv8 training flow. Export your chosen dataset from Roboflow in YOLOv8 format, upload it here, and verify `data.yaml` paths and class names before running.

Training can take time and is **not required** for the ready-to-run local app, which already includes trained weights. Do not publish API keys in notebooks.

In [ ]:
!pip -q install ultralytics pyyaml

## Upload the exported dataset

Upload a YOLOv8-format dataset ZIP and extract it into `/content/dataset`. If the exporter saves `data.yaml` outside the archive, upload that file too. Then edit the following paths to match your exported folder structure.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
for filename in uploaded:
    if filename.lower().endswith('.zip'):
        with zipfile.ZipFile(filename) as zf:
            zf.extractall('/content/dataset')
print('Dataset files:', list(Path('/content/dataset').glob('**/data.yaml'))[:5])


## Configure and train

Open the `data.yaml` file found above and ensure `train`, `val`, and optional `test` point to directories containing images. Dataset class names must match the annotation IDs. This template intentionally does not guess paths or label names.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# CHANGE this to the actual data.yaml path printed in the previous cell.
DATA_YAML = "/content/dataset/data.yaml"
assert Path(DATA_YAML).is_file(), f"Update DATA_YAML to the uploaded dataset YAML: {DATA_YAML}"

model = YOLO("yolov8n.pt")
results = model.train(data=DATA_YAML, epochs=25, imgsz=640, project="/content/runs/train", name="spill")
print("Best checkpoint:", Path(results.save_dir) / "weights" / "best.pt")


## Evaluate on labeled hold-out data

Use an untouched `test` split if your YAML defines it. Report metrics emitted by the evaluation run; do not equate confidence scores with accuracy.

In [ ]:
model = YOLO(str(Path(results.save_dir) / "weights" / "best.pt"))
metrics = model.val(data=DATA_YAML, split="test", imgsz=640)
print("Box metrics:", metrics.box)


## Export the checkpoint

Download the model weight to your local project and back up the current model before replacing it. The project `scripts/train.py --register` provides a registry-aware local training path when the exported dataset is on your computer.